# US 4.1 -- Multi-Site Data Analysis

Before training a domain-adaptive model we need to understand *how* domains differ.
This notebook shows how to:

1. Configure domain paths in `epic4_data.yaml`
2. Load datasets using `DomainDataset`
3. Compute intensity histograms per domain
4. Visualise samples from different domains

## CLI equivalent

```bash
udm-epic4 prepare --config configs/epic4_data.yaml
```

The CLI command runs the same analysis and prints a summary table to the terminal.

---
## 1. Data Configuration

All domain paths are specified in `configs/epic4_data.yaml`.  Here is the structure:

```yaml
domains:
  source:
    name: warstein
    images: data/warstein/images
    masks: data/warstein/masks
    train_ratio: 0.8
    val_ratio: 0.2

  targets:
    - name: malaysia
      images: data/malaysia/images
      # no masks -- unlabeled for DANN training

  evaluation:
    - name: warstein_val
      images: data/warstein/val/images
      masks: data/warstein/val/masks
    - name: malaysia_eval
      images: data/malaysia_eval/images
      masks: data/malaysia_eval/masks

image:
  height: 512
  width: 512
```

Key points:
- **Source domain** has both images and masks (fully labeled).
- **Target domains** may be unlabeled (`masks: null`) -- DANN only needs the images.
- **Evaluation domains** have masks so we can measure metrics after training.

In [ ]:
import yaml
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch

from udm_epic4.data.multi_domain_dataset import DomainDataset, build_datasets_from_config

In [ ]:
# Load the data configuration
config_path = Path("../configs/epic4_data.yaml")
with open(config_path) as f:
    cfg = yaml.safe_load(f)

print("Domains configured:")
print(f"  Source:     {cfg['domains']['source']['name']}")
for t in cfg['domains'].get('targets', []):
    print(f"  Target:     {t['name']}")
for e in cfg['domains'].get('evaluation', []):
    print(f"  Evaluation: {e['name']}")

---
## 2. Loading Datasets with DomainDataset

`DomainDataset` loads grayscale images from a directory, resizes them to
`image_size`, normalises with ImageNet stats (replicated to 3 channels),
and assigns an integer domain label.

Each sample is a dict with keys: `image` (tensor), `mask` (tensor or None),
`domain` (int), `path` (str).

In [ ]:
# Load individual domains
img_h = cfg['image']['height']
img_w = cfg['image']['width']
image_size = (img_h, img_w)

src_cfg = cfg['domains']['source']
source_ds = DomainDataset(
    images_dir=src_cfg['images'],
    masks_dir=src_cfg.get('masks'),
    domain_label=0,
    image_size=image_size,
)
print(f"Source dataset: {source_ds}")
print(f"  Number of samples: {len(source_ds)}")

# Load target domains
target_datasets = []
for i, t_cfg in enumerate(cfg['domains'].get('targets', [])):
    ds = DomainDataset(
        images_dir=t_cfg['images'],
        masks_dir=t_cfg.get('masks'),
        domain_label=i + 1,
        image_size=image_size,
    )
    target_datasets.append(ds)
    print(f"Target '{t_cfg['name']}': {len(ds)} images")

In [ ]:
# Inspect a single sample
if len(source_ds) > 0:
    sample = source_ds[0]
    print(f"Sample keys:   {list(sample.keys())}")
    print(f"Image shape:   {sample['image'].shape}")   # [3, H, W]
    print(f"Mask shape:    {sample['mask'].shape if sample['mask'] is not None else 'None'}")
    print(f"Domain label:  {sample['domain']}")
    print(f"Image path:    {sample['path']}")
else:
    print("Source dataset is empty -- check your data paths in epic4_data.yaml.")

---
## 3. Intensity Histograms per Domain

A quick way to see domain shift is to compare pixel intensity distributions.
Different X-ray machines produce different brightness/contrast profiles.

In [ ]:
def compute_intensity_histogram(dataset, n_samples=50, n_bins=100):
    """Compute average intensity histogram from the first channel of normalised images."""
    n = min(n_samples, len(dataset))
    all_values = []
    for i in range(n):
        img = dataset[i]['image'][0].numpy()  # first channel
        all_values.append(img.flatten())
    all_values = np.concatenate(all_values)
    hist, bin_edges = np.histogram(all_values, bins=n_bins, density=True)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    return bin_centers, hist


# Collect all domains for plotting
all_domains = [(src_cfg['name'], source_ds)]
for i, t_cfg in enumerate(cfg['domains'].get('targets', [])):
    if i < len(target_datasets) and len(target_datasets[i]) > 0:
        all_domains.append((t_cfg['name'], target_datasets[i]))

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#2196F3', '#FF5722', '#4CAF50', '#9C27B0']

for idx, (name, ds) in enumerate(all_domains):
    if len(ds) == 0:
        continue
    centers, hist = compute_intensity_histogram(ds)
    ax.plot(centers, hist, label=name, color=colors[idx % len(colors)], linewidth=2)

ax.set_xlabel("Normalised pixel intensity", fontsize=12)
ax.set_ylabel("Density", fontsize=12)
ax.set_title("Intensity Distribution per Domain", fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

print("Interpretation: if curves are shifted or have different shapes,")
print("there is domain shift that DANN needs to handle.")

---
## 4. Visualise Samples from Different Domains

Side-by-side comparison of images from each domain helps build intuition
about what kind of shift exists.

In [ ]:
def denormalise_for_display(img_tensor):
    """Convert a normalised [3,H,W] tensor back to displayable [H,W] grayscale."""
    img = img_tensor[0].numpy()  # take first channel
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    return img


n_samples_to_show = 3
n_domains = len(all_domains)

if n_domains > 0:
    fig, axes = plt.subplots(
        n_domains, n_samples_to_show,
        figsize=(4 * n_samples_to_show, 4 * n_domains),
    )
    if n_domains == 1:
        axes = axes[np.newaxis, :]  # ensure 2D indexing

    for row, (name, ds) in enumerate(all_domains):
        for col in range(n_samples_to_show):
            if col < len(ds):
                sample = ds[col]
                img = denormalise_for_display(sample['image'])
                axes[row, col].imshow(img, cmap='gray')
                axes[row, col].set_title(f"{name} #{col}", fontsize=10)
            axes[row, col].axis('off')

    fig.suptitle("Sample Images per Domain", fontsize=14, y=1.01)
    fig.tight_layout()
    plt.show()
else:
    print("No domains with data found. Check your config paths.")

---
## 5. Using `build_datasets_from_config` (DANN Training Pipeline)

For actual training, the `build_datasets_from_config` factory creates train/val
splits from the source domain and wraps target domains as separate datasets.
This is what the training scripts use internally.

The config format expected by this function is slightly different from
`epic4_data.yaml` -- it uses a `data:` key with `source:`, `targets:`,
and `evaluation:` sub-keys (as in `epic4_dann.yaml`).

```python
# Example: build datasets from the DANN training config
with open('configs/epic4_dann.yaml') as f:
    dann_cfg = yaml.safe_load(f)

datasets = build_datasets_from_config(dann_cfg['data'])
# datasets['source_train']  -- Subset for training
# datasets['source_val']    -- Subset for validation
# datasets['targets']       -- list of DomainDataset (unlabeled)
# datasets['evaluation']    -- list of DomainDataset (labeled)
```

---
## Summary

- Domain paths are configured in `configs/epic4_data.yaml`.
- `DomainDataset` loads images, applies normalisation, and assigns domain labels.
- Intensity histograms reveal distribution shift between domains.
- Visual inspection helps identify the nature of domain differences (brightness,
  contrast, noise, geometry).

**Next:** `epic4_02_baseline.ipynb` -- train a source-only model to see the
domain gap in action.